In [ ]:
import json
import requests
from openai import OpenAI
from tqdm import tqdm

# Initialize client
client = OpenAI(
    api_key="",
    base_url="https://api.deepseek.com"
)


In [2]:
system_prompt = """You are a medical expert assistant.
You are given multiple-choice medical questions.
Your task is to choose the single best answer.

OUTPUT CONSTRAINTS (STRICT):
- Do NOT provide any explanation, reasoning, or commentary.
- Do NOT include any words, symbols, or punctuation.
- Do NOT include quotes, prefixes, or suffixes.
- Do NOT include extra whitespace or newlines.
- The response must consist of exactly one character.

PENALTIES FOR VIOLATION:
- Any output longer than one character will be considered a critical error.
- Any output not exactly one of (A, B, C, D, or E) will receive a zero score.

FINAL FORMAT (MANDATORY):
X
where X is one of (A, B, C, D, or E)."""


user_prompt = """Question:
{question}

Options:
{options}

Select the single best answer.
Output exactly one character and nothing else:
A, B, C, D, or E."""

In [3]:
def form_message(question_text, options_dict):
  chat = {'conversations': []}
  chat['conversations'].append({
        "role": "system",
        "content": (
            system_prompt
        )
    })


  options_str = ''
  for k, v in options_dict.items():
      options_str += f'{k}. {v}\n'

  chat['conversations'].append({
        "role": "user",
        "content": (
            user_prompt.format(question=question_text,
                               options=options_str)
        )
    })
    
  return chat

# Inference 

In [4]:
import json
with open('/kaggle/input/medqa-us-data/final_augment_test_questions_v2.json', 'r') as f:
    dataset = json.loads(f.read())
    f.close()

In [5]:
dataset[0]

{'original_question': 'Two weeks after undergoing an emergency cardiac catherization with stenting for unstable angina pectoris, a 61-year-old man has decreased urinary output and malaise. He has type 2 diabetes mellitus and osteoarthritis of the hips. Prior to admission, his medications were insulin and naproxen. He was also started on aspirin, clopidogrel, and metoprolol after the coronary intervention. His temperature is 38°C (100.4°F), pulse is 93/min, and blood pressure is 125/85 mm Hg. Examination shows mottled, reticulated purplish discoloration of the feet. Laboratory studies show:\nHemoglobin count 14 g/dL\nLeukocyte count 16,400/mm3\nSegmented neutrophils 56%\nEosinophils 11%\nLymphocytes 31%\nMonocytes 2%\nPlatelet count 260,000/mm3\nErythrocyte sedimentation rate 68 mm/h\nSerum\nUrea nitrogen 25 mg/dL\nCreatinine 4.2 mg/dL\nRenal biopsy shows intravascular spindle-shaped vacuoles. Which of the following is the most likely cause of this patient\'s symptoms?"',
 'options': {'

In [6]:
all_chats = []
for d in dataset:
    chats = {}
    chats['original'] = form_message(d['original_question'], d['options'])['conversations']
    chats['neutral'] = form_message(d['neutral_question'], d['options'])['conversations']
    for t in ['t1', 't2', 't3']:
        chat_obj = {}
        for e in ['i', 'c', 'ic']:
            chat_obj[e] = form_message(d[t][e], d['options'])['conversations']
        chats[t] = chat_obj.copy()
            
    all_chats.append(chats)

In [7]:
def get_response(chat_data):
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=chat_data
    )

    reasoning = response.choices[0].message.reasoning_content
    final_answer = response.choices[0].message.content

    resp_data = {'reasoning_content': reasoning, 'content': final_answer}
    return resp_data

In [8]:
r= get_response(all_chats[0]['t1']['i'])
r

{'reasoning_content': "First, the patient is a 61-year-old First Nations man with a history of type 2 diabetes and osteoarthritis. He recently had emergency cardiac catheterization with stenting for unstable angina two weeks ago. Now, he presents with decreased urinary output and malaise, suggesting renal issues.\n\nMedications: Prior to admission, he was on insulin and naproxen. After the coronary intervention, he started aspirin, clopidogrel, and metoprolol.\n\nVital signs: Temperature 38°C (fever), pulse 93/min, BP 125/85 mm Hg.\n\nExamination: Mottled, reticulated purplish discoloration of the feet – this sounds like livedo reticularis, which can be associated with certain conditions.\n\nLab findings: \n- Hemoglobin: 14 g/dL (normal)\n- Leukocyte count: 16,400/mm3 (elevated, indicating infection or inflammation)\n- Differential: Segmented neutrophils 56% (normal or slightly low percentage but absolute count might be high due to high WBC), eosinophils 11% (elevated, normal is usuall

In [9]:
all_answers = []
for chats in tqdm(all_chats):
    ans_obj = {}
    ans_obj['original'] = get_response(chats['original'])
    ans_obj['neutral'] = get_response(chats['neutral'])
    for t in ['t1', 't2', 't3']:
        ans_obj[t] = {}
        for e in ['i', 'c', 'ic']:
            ans_obj[t][e] = get_response(chats[t][e])
    all_answers.append(ans_obj)
    
    if len(all_answers) % 10 == 0:
        with open('DeepSeekR1_option_augment_answers.json', 'w') as f:
            f.write(json.dumps(all_answers))
            f.close()


  0%|          | 0/150 [00:00<?, ?it/s]


  1%|          | 1/150 [04:05<10:09:41, 245.51s/it]


  1%|▏         | 2/150 [30:16<42:09:07, 1025.32s/it]


  2%|▏         | 3/150 [33:02<25:50:23, 632.81s/it] 


  3%|▎         | 4/150 [37:52<20:10:44, 497.56s/it]

...


 75%|███████▍  | 112/150 [11:29:51<3:24:37, 323.08s/it]


 75%|███████▌  | 113/150 [11:36:06<3:28:50, 338.65s/it]


 76%|███████▌  | 114/150 [11:39:28<2:58:28, 297.46s/it]


 77%|███████▋  | 115/150 [11:44:55<2:58:41, 306.33s/it]


 77%|███████▋  | 116/150 [11:55:18<3:47:26, 401.36s/it]

In [ ]:
with open('DeepSeekR1_option_augment_answers.json', 'w') as f:
    f.write(json.dumps(all_answers))
    f.close()